# Train and Validate
* Train a model across the specified training sites and sampling approach
* Validate the model at the excluded site

## Todo
* Pull out probabilities using model.predict_proba(observations_to_predict)

In [1]:
import pathlib
import numpy
import dask.distributed
import pandas
import joblib

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import plotting

%load_ext autoreload
%autoreload 2

In [2]:
cluster = dask.distributed.LocalCluster(
    threads_per_worker=1,   n_workers=4,                                 
    config={
        "distributed.worker.memory.target": 0.80, 
        "distributed.worker.memory.spill": 0.85,
        "distributed.worker.memory.pause": 0.90
    }
)
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 4,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:54807,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:54836,Total threads: 1
Dashboard: http://127.0.0.1:54838/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:54810,


# Values to edit

In [21]:
all_training_sites =  ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

baseline_sites = all_training_sites

import sentinel2
baseline_max_cloud_cover = 0
baseline_tide_deltas = {"30mins": {"hrs": 0, "mins": 30},
                        "45mins": {"hrs": 0, "mins": 45},
                        "1hr": {"hrs": 1, "mins": 0}}

satellite_paths = {}
for tide_delta_name, tide_delta in baseline_tide_deltas.items():
    predictions_path, satellite_path = utils.get_prediction_path(
        sample_method="sampling_2", method_2_threshold=0.5,
        model_max_cloud_cover=baseline_max_cloud_cover,
        predict_max_cloud_cover=baseline_max_cloud_cover,
        predict_low_tide_delta_hrs=tide_delta["hrs"],
        predict_low_tide_delta_mins=tide_delta["mins"],
        model_low_tide_delta_hrs=tide_delta["hrs"], # dummy value as models not considered this notebook
        model_low_tide_delta_mins=tide_delta["mins"]) # dummy value as models not considered this notebook
    satellite_paths[tide_delta_name] = satellite_path

baseline = sentinel2.compute_and_save_band_quality_baseline(
    training_sites=baseline_sites[0:9],
    satellite_images_paths=list(satellite_paths.values()),
)
baseline

	Saved band quality baseline for 9 sites to C:\Local\repos\seagrass-detection\data\band_quality_baseline.csv


,site,band,median,mad,mad_raw,n_images
0,Ihutai,B04,502.00,48.975330,8.00,16
1,Ihutai,B03,642.00,50.737600,31.00,16
2,Ihutai,B02,568.00,45.043712,29.00,16
3,CatlinsLake,B04,356.00,55.858741,45.50,50
4,CatlinsLake,B03,378.00,59.007908,56.00,50
5,CatlinsLake,B02,262.00,47.965337,44.00,50
6,CatlinsRiverMouth,B04,814.00,92.201492,96.00,55
7,CatlinsRiverMouth,B03,877.00,80.807381,86.00,55
8,CatlinsRiverMouth,B02,637.00,46.234315,42.00,55
9,Childrens,B04,423.00,81.972692,82.00,159


In [3]:
data_path = utils.get_data_path()

# Cells to run
* Lift out all confusion matricies for different low_tide_delta and max_cloud_percent and plot in a single file

In [6]:
survey_date_site_groups = {
    "IveyBay_Nov25": "IveyBay",
    "IveyBay_Feb26": "IveyBay",
    "LeftBank_Nov25": "LeftBank",
    "LeftBank_Feb26": "LeftBank",
    "Paremata_Nov25": "Paremata",
    "Paremata_Feb26": "Paremata",
    "Paremata_Feb25": "Paremata",
    "ThePoint_Nov25": "ThePoint",
    "ThePoint_Feb26": "ThePoint",
    "Takapuwahia_Nov25": "Takapuwahia",
    "Takapuwahia_Feb26": "Takapuwahia",
}

plotting.plot_site_predicted_vs_surveyed_areas(
    data_path=data_path,
    survey_date_site_groups=survey_date_site_groups,
    include_rf=True, include_unet=True,
)

Found model folder: C:\Local\repos\seagrass-detection\data\website\RF_model_all_trained_13_class_10_percent_test_train_sampling_2_50_percent_tide_30mins_cloud_10_predict_tide_30mins_cloud_1
Found model folder: C:\Local\repos\seagrass-detection\data\website\RF_model_all_trained_13_class_10_percent_test_train_sampling_2_50_percent_tide_30mins_cloud_1_predict_tide_30mins_cloud_1
Found model folder: C:\Local\repos\seagrass-detection\data\website\RF_model_all_trained_13_class_10_percent_test_train_sampling_2_50_percent_tide_30mins_cloud_5_predict_tide_30mins_cloud_1
Found model folder: C:\Local\repos\seagrass-detection\data\website\RF_model_all_trained_13_class_10_percent_test_train_sampling_2_80_percent_tide_30mins_cloud_1_predict_tide_30mins_cloud_1
Found model folder: C:\Local\repos\seagrass-detection\data\website\RF_model_all_trained_13_class_10_percent_test_train_sampling_2_80_percent_tide_30mins_cloud_5_predict_tide_30mins_cloud_1
Found model folder: C:\Local\repos\seagrass-detection\

[WindowsPath('C:/Local/repos/seagrass-detection/data/website/site_area_timeseries/CatlinsLake_target_area_timeseries.html'),
 WindowsPath('C:/Local/repos/seagrass-detection/data/website/site_area_timeseries/CatlinsRiverMouth_target_area_timeseries.html'),
 WindowsPath('C:/Local/repos/seagrass-detection/data/website/site_area_timeseries/Childrens_target_area_timeseries.html'),
 WindowsPath('C:/Local/repos/seagrass-detection/data/website/site_area_timeseries/Duvauchelle_target_area_timeseries.html'),
 WindowsPath('C:/Local/repos/seagrass-detection/data/website/site_area_timeseries/Ihutai_target_area_timeseries.html'),
 WindowsPath('C:/Local/repos/seagrass-detection/data/website/site_area_timeseries/IveyBay_target_area_timeseries.html'),
 WindowsPath('C:/Local/repos/seagrass-detection/data/website/site_area_timeseries/IveyBay_ThePoint_LeftBank_Oct24_target_area_timeseries.html'),
 WindowsPath('C:/Local/repos/seagrass-detection/data/website/site_area_timeseries/LeftBank_target_area_timeser

In [5]:
method_2_threshold = .5 # 1.0 .99 .98 .97 .95 .90
test_threshold = 0.1
model_validation_label = f"RF_model_all_trained_13_class_{int(test_threshold*100)}_percent_test"

plotting.plot_validation_confusion_matrices(
    data_path=data_path,
    model_validation_label=model_validation_label,
    method_2_threshold=method_2_threshold,
)

WindowsPath('C:/Local/repos/seagrass-detection/data/validation/overall_plots/sampling_2_50_percent_RF_model_all_trained_13_class_10_percent_test_confusion_matrix.png')